# MODIS Terra L1B Extraction

This notebook demonstrates how to search for MODIS Terra L1B granules via **earthaccess** and extract visible bands using **satpy**.

| Property | Value |
|----------|-------|
| 📡 Sensor | MODIS Terra L1B |
| ⏱ Estimated time | ~5 min |
| 💾 Disk usage | ~800 MB |
| 🔐 Auth required | Earthdata 🔐 |

> 📖 New to AER? Read the [main README](../../README.md) for the full quickstart and API overview.

## Setup

Import the required libraries and configure the search parameters.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import time

import geopandas as gpd

from aer.client import AerClient
from aer.interfaces import ExtractionProfile
from aer.search_earthaccess import earthaccess_download_wrapper

## Configuration & Earthdata Authentication

MODIS Terra L1B granules are hosted by NASA and require **Earthdata authentication**.

`earthaccess.login()` handles credential caching (`.netrc` or environment variables). If you have not configured credentials, run `earthaccess.login()` interactively once to store them.

`plugin_hints={'search_earthaccess': collections}` forces AER to use the `search_earthaccess` plugin, which wraps NASA's CMR (Common Metadata Repository) API. Without the hint, AER would still resolve it automatically because `search_earthaccess` declares `MOD021KM` in its `supported_collections`.

## Configuration

Define the AOI, time range, and collection.

In [ ]:
# --- Configuration ---
DATE_START = datetime(2025, 12, 10, 0, 0, tzinfo=timezone.utc)
DATE_END = datetime(2025, 12, 20, 0, 0, tzinfo=timezone.utc)

# Load AOI (Buenos Aires province)
geojson_path = Path("../neuquen_city.geojson")
gdf = gpd.read_file(geojson_path)
aoi = gdf.geometry.iloc[0]

# MODIS Terra L1B 1km collection
collections = ["MOD021KM"]

# --- Client Setup ---
client = AerClient()

## ExtractionProfile in Detail

`ExtractionProfile` is an `@attrs.frozen` dataclass defined in `aer.interfaces.core`. Key fields:

1. **`name`** — Arbitrary label for your own bookkeeping.
2. **`resolution`** — Target pixel size in metres. MODIS 1km native resolution is 1000 m; we extract at 1000 m.
3. **`collection_variables_map`** — Dict mapping each collection to the list of bands/variables you want.
   Here we ask for band `1` (red visible) from `MOD021KM`.
4. **`reader`**, **`padding`**, **`resampling`**, **`calibration`**, **`satellite`** — Domain-specific extraction parameters.
5. **`extra_params`** — Fallback for any additional plugin-specific key/value pairs.

You can define multiple profiles in one run (e.g. one for thermal, one for optical). Each profile produces its own set of tasks and output files.

## Search

Search for MODIS granules that intersect the AOI using `search_earthaccess`.

In [ ]:
print("Searching...", flush=True)
results = client.search(
    collections=collections,
    start_datetime=DATE_START,
    end_datetime=DATE_END,
    intersects=aoi,
    plugin_hints={"search_earthaccess": collections},
)
print(f"Found {len(results)} results", flush=True)
results.head()

## Extraction Parameters & Download Wrapper

`extract_params` is reserved for meta-level or tool-level parameters (e.g. `downloader`).

Because MODIS data is **not** public S3, we must provide a `downloader`. `earthaccess_download_wrapper` is a thin wrapper around `earthaccess.download()` that conforms to AER's `Downloader` protocol: it accepts `(url, local_path)` and blocks until the file is written. The extractor calls it for every granule URL.

Domain-specific configuration (`padding`, `resampling`, `calibration`, `reader`) lives on the `ExtractionProfile` itself.

`max_batch_workers=4` runs four processes in parallel. Use `None` for sequential execution (lower memory).

## Prepare Extraction

Define extraction profiles and prepare tasks.

In [ ]:
# --- Prepare Extraction ---
uri = "extract_neuquen_modis"

profiles = [
    ExtractionProfile(
        name="modis_thermal",
        resolution=1000,
        collection_variables_map={"MOD021KM": ["1"]},
        reader="modis_l1b",
        padding=2,
        resampling="nearest",
        calibration="reflectance",
    )
]

prepare_params = {"cells_per_chunk": 10}

tasks = client.prepare_for_extraction(
    results,
    target_aoi=aoi,
    uri=uri,
    profiles=profiles,
    target_grid_dist=256000,
    target_grid_overlap=False,
    prepare_params=prepare_params,
    plugin_hints={"extract_satpy": collections},
)

print(f"Prepared {len(tasks)} extraction tasks", flush=True)

## Extract

Run the extraction pipeline.

In [ ]:
# Clean output directory
uri_path = Path(uri)
if uri_path.exists():
    shutil.rmtree(uri_path)
uri_path.mkdir(parents=True)

print("Extracting...", flush=True)
start_time = time.time()

# extract_params is reserved for meta-level / tool-level parameters.
# Domain-specific config (padding, calibration, reader, etc.) lives on the profile.
extract_params = {
    "downloader": earthaccess_download_wrapper,
}

results_df = client.extract_batches(
    tasks,
    extract_params=extract_params,
    plugin_hints={"extract_satpy": collections},
    max_batch_workers=4,
)

end_time = time.time()
print(f"Extraction took {end_time - start_time:.2f}s")
print(f"Extracted {len(results_df)} artifacts")

## Results

Inspect the extracted artifacts.

In [ ]:
results_df[["id", "collection", "grid_cell", "uri"]].head()